<a href="https://colab.research.google.com/github/yuhan222/114-2-Programing-Language/blob/main/HW4_PTT_GoogleSheet_RAG_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：PTT → Google Sheet → RAG

完整功能清單：

1. 基本設定
2. Gemini 設定
3. 連線 Google Sheet
4. PTT movie 爬蟲
5. 執行爬蟲並寫入 Google Sheet
6. 🆕 文章自動摘要（存回 Google Sheet）
7. 🆕 推文情緒分析（存回 Google Sheet）
8. 從 Google Sheet 建立 FAISS RAG 索引
9. 🆕 多輪對話問答（含回答信心分數）
10. 🆕 問答紀錄存回 Google Sheet
11. 🆕 Gradio 互動介面
12. 啟動問答


In [1]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 40.3 MB/s eta 0:00:00


In [2]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。  
`PTT_WORKSHEET_NAME` 是存放 PTT 原始文章的分頁。  
`SUMMARY_WORKSHEET_NAME` 🆕 是存放文章摘要的分頁。

In [3]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1xBFqAnS-W1WzXEqhYGcMkoCYZ295mOI_j94tppr-H2o/edit?usp=sharing"
PTT_WORKSHEET_NAME = "ptt_movie_posts"
SUMMARY_WORKSHEET_NAME = "ptt_movie_summaries"   # 摘要分頁
SENTIMENT_WORKSHEET_NAME = "ptt_movie_sentiment"  # 🆕 情緒分析分頁
QA_LOG_WORKSHEET_NAME = "ptt_qa_log"              # 🆕 問答紀錄分頁
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]
SUMMARY_HEADER = ["post_id", "title", "url", "summary", "summarized_at"]
SENTIMENT_HEADER = ["post_id", "title", "url", "sentiment", "reason", "analyzed_at"]  # 🆕
QA_LOG_HEADER = ["session_id", "turn", "question", "answer", "sources", "confidence", "logged_at"]  # 🆕

PTT_MOVIE_INDEX = "https://www.ptt.cc/bbs/movie/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (compatible; Colab PTT crawler)"


## 2. Gemini 設定

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。
Gemini 需要在最前面設定好，後續的摘要與問答功能才能正常使用。

In [24]:
api_key = userdata.get("gemini")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

GEMINI_MODEL_NAME = "gemini-2.5-flash"  # 請依你的帳號可用模型修改
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


✅ Gemini 已設定：gemini-2.5-flash


## 3. 連線 Google Sheet

In [5]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")

✅ 已開啟試算表：程式語言作業範例測試資料 的副本
🔗 https://docs.google.com/spreadsheets/d/1xBFqAnS-W1WzXEqhYGcMkoCYZ295mOI_j94tppr-H2o/edit?usp=sharing


In [6]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("")
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)
    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)



ws_ptt      = ensure_worksheet(sh, PTT_WORKSHEET_NAME,       PTT_HEADER)
ws_summary  = ensure_worksheet(sh, SUMMARY_WORKSHEET_NAME,   SUMMARY_HEADER)
ws_sentiment = ensure_worksheet(sh, SENTIMENT_WORKSHEET_NAME, SENTIMENT_HEADER)  # 🆕
ws_qa_log   = ensure_worksheet(sh, QA_LOG_WORKSHEET_NAME,    QA_LOG_HEADER)      # 🆕
print(f"✅ 已準備 worksheet：{ws_ptt.title}")
print(f"✅ 已準備摘要 worksheet：{ws_summary.title}")
print(f"✅ 已準備情緒分析 worksheet：{ws_sentiment.title}")  # 🆕
print(f"✅ 已準備問答紀錄 worksheet：{ws_qa_log.title}")    # 🆕


✅ 已準備 worksheet：ptt_movie_posts
✅ 已準備摘要 worksheet：ptt_movie_summaries
✅ 已準備情緒分析 worksheet：ptt_movie_sentiment
✅ 已準備問答紀錄 worksheet：ptt_qa_log


## 4. PTT movie 爬蟲

In [7]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_soup(url):
    resp = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": USER_AGENT},
        cookies=PTT_COOKIES,
    )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆":
        return 100
    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10
    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a:
            continue
        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))
        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")
        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main:
        return "", ""
    created_at = ""
    metalines = main.select("div.article-metaline")
    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()
    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at


def make_post_id(url):
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_movie(pages=2, delay=0.5):
    """爬取 PTT movie 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_MOVIE_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }
                all_rows.append(row)
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")

        prev_url = get_prev_index_url(index_soup)
        if not prev_url:
            break
        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次爬到 {len(df)} 篇文章")
    return df

## 5. 執行爬蟲並寫入 Google Sheet

In [9]:
new_posts_df = crawl_ptt_movie(pages=2, delay=1.0)

old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)

written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")

📄 正在讀取列表頁 1/2: https://www.ptt.cc/bbs/movie/index.html
📄 正在讀取列表頁 2/2: https://www.ptt.cc/bbs/movie/index11001.html
✅ 本次爬到 39 篇文章
📌 Google Sheet 原本有 80 筆
✅ 已寫入 Google Sheet：80 筆
🔍 從 Google Sheet 重新讀回：80 筆
✅ 寫入驗證成功


## 6. 🆕 文章自動摘要並存回 Google Sheet

用 Gemini 為每篇文章產生一句話摘要，存入獨立的 `ptt_movie_summaries` 分頁。  
已有摘要的文章會自動跳過，不重複花費 API 呼叫。

## 7. 🆕 推文情緒分析並存回 Google Sheet

用 Gemini 判斷每篇文章的整體網友評價：**推薦 / 普通 / 負評**，並給出簡短理由。  
結果存入 `ptt_movie_sentiment` 分頁，RAG 回答時可參考情緒標籤。  
已分析過的文章自動跳過。


In [10]:
def analyze_sentiment(title, content, max_content_chars=800):
    """
    用 Gemini 判斷文章情緒：推薦 / 普通 / 負評。
    回傳 (sentiment, reason) tuple。
    """
    short_content = content[:max_content_chars]
    prompt = f"""請根據以下 PTT 電影版文章，判斷網友對這部電影或主題的整體評價。

只能從以下三個選項擇一：推薦、普通、負評
並用一句話（20字以內）說明理由。

請以以下格式回答（只輸出這兩行，不要其他文字）：
評價：<推薦/普通/負評>
理由：<一句話>

標題：{title}
內容：{short_content}
"""
    try:
        response = llm.generate_content(prompt)
        text = response.text.strip()
        sentiment = "普通"
        reason = ""
        for line in text.splitlines():
            if line.startswith("評價："):
                val = line.replace("評價：", "").strip()
                if val in ("推薦", "普通", "負評"):
                    sentiment = val
            elif line.startswith("理由："):
                reason = line.replace("理由：", "").strip()
        return sentiment, reason
    except Exception as e:
        return "失敗", str(e)


def batch_analyze_sentiment(posts_df, ws_sentiment, delay=1.5):
    """
    對所有文章做情緒分析，已分析的跳過，結果存回 ws_sentiment。
    """
    existing_df = read_sheet_df(ws_sentiment, SENTIMENT_HEADER)
    existing_ids = set(existing_df["post_id"].tolist())

    to_analyze = posts_df[
        ~posts_df["post_id"].isin(existing_ids) &
        (posts_df["content"].astype(str).str.strip() != "")
    ].copy()

    print(f"🎭 共 {len(posts_df)} 篇文章，需新增情緒分析：{len(to_analyze)} 篇")

    if to_analyze.empty:
        print("✅ 所有文章都已有情緒分析，略過。")
        return existing_df

    new_rows = []
    for i, (_, row) in enumerate(to_analyze.iterrows()):
        print(f"  [{i+1}/{len(to_analyze)}] 分析中：{row['title'][:30]}...")
        sentiment, reason = analyze_sentiment(row["title"], row["content"])
        new_rows.append({
            "post_id": row["post_id"],
            "title": row["title"],
            "url": row["url"],
            "sentiment": sentiment,
            "reason": reason,
            "analyzed_at": now_iso(),
        })
        time.sleep(delay)

    new_df = pd.DataFrame(new_rows, columns=SENTIMENT_HEADER)
    combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    combined_df = combined_df.drop_duplicates(subset=["post_id"], keep="last")
    write_sheet_df(ws_sentiment, combined_df, SENTIMENT_HEADER)
    print(f"✅ 情緒分析已寫入 Google Sheet，共 {len(combined_df)} 筆")
    return combined_df


# 執行情緒分析
sentiment_df = batch_analyze_sentiment(ptt_posts_df, ws_sentiment, delay=1.5)
sentiment_df.head()


🎭 共 80 篇文章，需新增情緒分析：40 篇
  [1/40] 分析中：[魚鉤雷] 漂流慾室（韓國電影，2000年）...
  [2/40] 分析中：[新聞] <妳最後留下的歌>道枝駿佑演繹少年普通感...
  [3/40] 分析中：[新聞] 話題恐怖片《後室》吸引Z世代進入電影院 Youtu...
  [4/40] 分析中：[新聞]元華曝成龍真實性格！與洪金寶對比鮮明！...
  [5/40] 分析中：[新聞]《後室》票房大賣1.18億美元！但續集電影...
  [6/40] 分析中：[好雷]一個妥瑞氏症患者看《出口成髒》...
  [7/40] 分析中：[新聞] 《橡樹街末日》預告 安海瑟薇對抗恐龍極限...
  [8/40] 分析中：[隱身雷] 空屋情人（韓國電影．2004年）...


  [9/40] 分析中：[討論] 大衛雷奇導演新作 《搶銀行先修班》預告...


  [10/40] 分析中：[新聞] 查克史奈德將執導《紐約大逃亡》新版電影...


  [11/40] 分析中：[討論] 微暴雷！屍速禁區最後眼球動的是誰...
  [12/40] 分析中：[普雷]《花綠青綻放之時》終將學會別離...
  [13/40] 分析中：[討論] 《九品芝麻官》包龍星與李連英對罵片段（粵語/國語/...
  [14/40] 分析中：[新聞] 配樂大師漢斯季默9月首來台...
  [15/40] 分析中：[請益] 007系列推薦補完清單...
  [16/40] 分析中：[好雷] The Life List (2025) 生命清單...
  [17/40] 分析中：Re: [討論] 台灣什麼時候能出現YouTuber拍電影的...
  [18/40] 分析中：[新聞]票房破億還不夠《大濛》空降Netflix榜首...
  [19/40] 分析中：[普無雷] 屍速禁區...
  [20/40] 分析中：[情報] 《外賣：4K數位紀念版》7/8 全台上映...
  [21/40] 分析中：[微負雷] 屍速禁區...
  [22/40] 分析中：[新聞]兩位極度強大的DC角色將在DCU宇宙登場！...
  [23/40] 分析中：[新聞]新任《黑豹》由誰接班？兩大熱門人選浮現！...


  [24/40] 分析中：[好雷] 紐倫堡...
  [25/40] 分析中：[討論]蓋兒加朵、艾爾帕西諾《但丁之手》預告！...


...

  [38/40] 分析中：Re: [討論] 庵野秀明:出淵裕真是心胸寬大之人......


  [39/40] 分析中：[負雷] 刀鋒戰士3 - 爆米花片都稱不上的爛...


  [40/40] 分析中：[贈票] 5月30日捐血送電影票！北捷西門站1號出口...


✅ 情緒分析已寫入 Google Sheet，共 80 筆


,post_id,title,url,sentiment,reason,analyzed_at
0,M.1780135252.A.19E,[新聞] 希區考克經典懸疑驚悚片重啟！《鳥》背景與角色設定全改,https://www.ptt.cc/bbs/movie/M.1780135252.A.19...,普通,陣容堅強且設定新穎，大改經典令影迷觀望。,2026-05-31 7:18:50
1,M.1780131947.A.10F,[LIVE] HBO 21:00 只為你遺憾,https://www.ptt.cc/bbs/movie/M.1780131947.A.10...,普通,文章僅分享播映資訊，未包含網友具體評價。,2026-05-31 7:18:59
2,M.1780118612.A.0B6,Re: [普微雷] 《後室》,https://www.ptt.cc/bbs/movie/M.1780118612.A.0B...,普通,氣氛雖具壓迫感但劇本邏輯難以自圓其說。,2026-05-31 7:19:05
3,M.1780112444.A.BC7,[新聞] 發錢！《九品芝麻官》李公公過世！劉洵,https://www.ptt.cc/bbs/movie/M.1780112444.A.BC...,推薦,影迷對其演技與經典配角地位給予高度肯定。,2026-05-31 7:19:13
4,M.1780110071.A.25B,[新聞] 伊藤潤二短篇《閣樓的長髮》改編亞洲恐怖,https://www.ptt.cc/bbs/movie/M.1780110071.A.25...,普通,內容僅為轉發改編計畫之新聞，未見網友討論。,2026-05-31 7:19:31


## 8. 從 Google Sheet 建立 FAISS RAG 索引

資料來源從 Google Sheet 重新讀回，摘要與情緒分析標籤也一併納入向量化文字。


In [7]:
rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

summary_source_df = read_sheet_df(ws_summary, SUMMARY_HEADER)
summary_map = dict(zip(summary_source_df["post_id"], summary_source_df["summary"]))
print(f"📝 已載入摘要筆數：{len(summary_map)}")

# 🆕 載入情緒分析
sentiment_source_df = read_sheet_df(ws_sentiment, SENTIMENT_HEADER)
sentiment_map = dict(zip(sentiment_source_df["post_id"],
                         sentiment_source_df["sentiment"] + "：" + sentiment_source_df["reason"]))
print(f"🎭 已載入情緒分析筆數：{len(sentiment_map)}")

rag_source_df.head()


📚 可用於 RAG 的文章數：80
📝 已載入摘要筆數：16
🎭 已載入情緒分析筆數：80


,post_id,title,url,date,author,nrec,created_at,fetched_at,content
0,M.1780335413.A.1E1,[普無雷] 《出口成髒》不建議看我這篇,https://www.ptt.cc/bbs/movie/M.1780335413.A.1E...,6/2,nobady98,0,Tue Jun 2 01:36:51 2026,2026-06-03 3:06:58,上週我趕不上最後一場\n以為沒得看了\n沒想到那只是口碑場\n\n劇情有點太過跳躍\n節奏忽...
1,M.1780334467.A.A2A,[討論] DCU 雷克斯路瑟動力裝甲片廠照,https://www.ptt.cc/bbs/movie/M.1780334467.A.A2...,6/2,labich,29,Tue Jun 2 01:21:05 2026,2026-06-03 3:06:57,http://i.imgur.com/PGVbCQs.jpg\nhttps://x.com/...
2,M.1780330284.A.B80,[ 好雷] 給阿嬤的情書,https://www.ptt.cc/bbs/movie/M.1780330284.A.B8...,6/2,Dissipate,3,Tue Jun 2 00:11:22 2026,2026-06-03 3:06:55,上週被老婆拉著去看潮汕方言版的給阿嬤的情書，整體劇本確實很感人，邏輯也通透，部份\n疑難細節...
3,M.1780328198.A.BB9,[新聞]拍《明日邊界》艾蜜莉布朗讚阿湯哥不一般！,https://www.ptt.cc/bbs/movie/M.1780328198.A.BB...,6/1,XDGEE,8,Mon Jun 1 23:36:36 2026,2026-06-03 3:06:54,拍《明日邊界》超折磨！艾蜜莉布朗讚阿湯哥不一般！\n最近，英國女演員艾蜜莉‧布朗回憶起在拍《...
4,M.1780328150.A.D43,[新聞]史匹柏談創意底線：不能讓AI成為決策者！,https://www.ptt.cc/bbs/movie/M.1780328150.A.D4...,6/1,XDGEE,8,Mon Jun 1 23:35:48 2026,2026-06-03 3:06:53,匹柏談創意底線：不能讓AI成為決策者！\n隨著好萊塢關於人工智慧（AI）的應用議題持續升溫，...


In [8]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")


def build_rag_documents(df, summary_map, sentiment_map={}):
    docs = []
    for _, row in df.iterrows():
        post_id = str(row.get("post_id", ""))
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))
        # 🆕 加入摘要欄位，摘要放在最前面有助於提升檢索相關性
        summary = summary_map.get(post_id, "")
        summary_line = f"摘要：{summary}\n" if summary else ""
        # 🆕 加入情緒標籤
        sentiment_info = sentiment_map.get(post_id, "")
        sentiment_line = f"網友評價：{sentiment_info}\n" if sentiment_info else ""

        text = (f"{summary_line}"
                f"{sentiment_line}"
                f"標題：{title}\n"
                f"作者：{author}\n"
                f"日期：{date}\n"
                f"推文數：{nrec}\n"
                f"內容：{content}")
        docs.append({
            "post_id": post_id,
            "title": title,
            "url": url,
            "summary": summary,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")
    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df, summary_map, sentiment_map)  # 🆕 傳入 summary_map 與 sentiment_map
rag_index, rag_embeddings = build_faiss_index(rag_documents)
print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding 模型載入完成


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

✅ RAG 索引建立完成：80 篇文章，向量維度 384


In [9]:
CONFIDENCE_THRESHOLDS = {
    "高": 1.0,   # distance < 1.0
    "中": 2.0,   # distance < 2.0
    "低": 999,   # 其他
}

def get_confidence_label(avg_distance):
    for label, threshold in CONFIDENCE_THRESHOLDS.items():
        if avg_distance < threshold:
            return label
    return "低"


def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    """單次問答（無記憶），含回答信心分數。"""
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關 PTT 資料。"

    # 🆕 計算信心分數
    avg_dist = sum(d["distance"] for d in docs) / len(docs)
    confidence = get_confidence_label(avg_dist)

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    answer = response.text

    # 🆕 附上信心分數
    sources = [d['title'] for d in docs]
    print(f"📊 資料充足度：{confidence}（平均距離 {avg_dist:.2f}，參考 {len(docs)} 篇文章）")
    return answer, confidence, sources


## 9. 🆕 多輪對話問答

支援上下文記憶，可以追問。每次回答附上資料充足度。  
輸入 `quit` 或 `exit` 結束，輸入 `reset` 清除記憶。


In [10]:
def build_chat_prompt(question, docs, history):
    """
    組合包含對話歷史的 prompt。
    history 格式：[{"role": "user"|"assistant", "content": str}, ...]
    """
    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    # 把歷史對話格式化成文字
    history_text = ""
    if history:
        lines = []
        for turn in history:
            role_label = "使用者" if turn["role"] == "user" else "助教"
            lines.append(f"【{role_label}】{turn['content']}")
        history_text = "\n".join(lines)

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}
""".strip()

    if history_text:
        prompt += f"\n\n【對話歷史】\n{history_text}"

    prompt += f"\n\n【使用者】{question}\n\n【助教】"
    return prompt


def chat_rag(question, history, k=3):
    """
    多輪對話問答。
    history：對話歷史清單，由外部維護並傳入。
    回傳：(answer_str, updated_history)
    """
    docs = retrieve_docs(question, k=k)
    confidence = "低"
    sources = []
    if not docs:
        answer = "找不到相關 PTT 資料。"
    else:
        # 🆕 信心分數
        avg_dist = sum(d["distance"] for d in docs) / len(docs)
        confidence = get_confidence_label(avg_dist)
        sources = [d["title"] for d in docs]
        prompt = build_chat_prompt(question, docs, history)
        response = llm.generate_content(prompt)
        answer = response.text.strip()

    updated_history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return answer, confidence, sources, updated_history


def start_chat(k=3, max_history_turns=6):
    """
    互動式多輪對話迴圈。
    max_history_turns：保留最近幾輪對話，避免 prompt 過長。
    """
    import uuid as _uuid
    session_id = str(_uuid.uuid4())[:8]
    turn_count = 1
    print("🎬 PTT 電影版 RAG 多輪對話（輸入 reset 清除記憶，quit 結束）")
    print(f"🔑 本次 Session ID：{session_id}")
    print("-" * 60)
    history = []

    while True:
        try:
            question = input("\n你：").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 對話結束。")
            break

        if not question:
            continue
        if question.lower() in ("quit", "exit", "bye"):
            print("👋 對話結束。")
            break
        if question.lower() == "reset":
            history = []
            turn_count = 1
            session_id = str(_uuid.uuid4())[:8]
            print(f"🔄 對話記憶已清除，新 Session ID：{session_id}")
            continue

        # 只保留最近 max_history_turns 輪，避免 prompt 太長
        recent_history = history[-(max_history_turns * 2):]

        answer, confidence, sources, history = chat_rag(question, recent_history, k=k)
        print(f"\n助教：{answer}")
        print(f"📊 資料充足度：{confidence}，參考文章：{len(sources)} 篇")
        print("-" * 60)
        # 🆕 存問答紀錄
        log_qa(session_id, turn_count, question, answer, sources, confidence)
        turn_count += 1

def log_qa(session_id, turn, question, answer, sources, confidence):
    """
    將單筆問答紀錄寫入 Google Sheet 的 ptt_qa_log 分頁。
    """
    existing_df = read_sheet_df(ws_qa_log, QA_LOG_HEADER)
    new_row = pd.DataFrame([{
        "session_id": session_id,
        "turn": str(turn),
        "question": question,
        "answer": answer,
        "sources": " | ".join(sources),
        "confidence": confidence,
        "logged_at": now_iso(),
    }], columns=QA_LOG_HEADER)
    combined = pd.concat([existing_df, new_row], ignore_index=True)
    write_sheet_df(ws_qa_log, combined, QA_LOG_HEADER)

print("✅ log_qa 與對話函式已載入")


✅ log_qa 與對話函式已載入


## 10. 🆕 問答紀錄存回 Google Sheet

`log_qa` 函式已整合至第 9 節一起載入，對話時自動記錄至 `ptt_qa_log` 分頁。


In [11]:
print("✅ 問答紀錄功能已在第 9 節一起載入。")


✅ 問答紀錄功能已在第 9 節一起載入。


## 11. 🆕 Gradio 互動介面

提供網頁介面讓你用滑鼠操作，不需要在 input() 輸入文字。  
功能包含：
- **多輪對話**：對話視窗，支援追問，自動記錄至 Google Sheet
- **資料充足度**：每次回答顯示信心分數
- **清除對話**：一鍵重置記憶


In [25]:
import time
import re as _re

def chat_rag(question, history, k=3):
    """多輪對話問答，含自動重試機制。"""
    docs = retrieve_docs(question, k=k)
    confidence = "低"
    sources = []
    if not docs:
        answer = "找不到相關 PTT 資料。"
    else:
        avg_dist = sum(d["distance"] for d in docs) / len(docs)
        confidence = get_confidence_label(avg_dist)
        sources = [d["title"] for d in docs]
        prompt = build_chat_prompt(question, docs, history)
        max_retries = 5
        answer = "[未取得回答]"
        for attempt in range(max_retries):
            try:
                print(f"🔄 呼叫 Gemini（第 {attempt+1} 次）...")
                response = llm.generate_content(prompt)
                answer = response.text.strip()
                print("✅ 取得回答")
                break
            except Exception as e:
                err = str(e)
                print(f"⚠️ 錯誤：{err[:200]}")
                if "429" in err or "quota" in err.lower() or "resource" in err.lower():
                    match = _re.search(r"retry in ([\d.]+)s", err)
                    wait = float(match.group(1)) + 2 if match else 60
                    print(f"⏳ 限流，等待 {wait:.0f} 秒後重試...")
                    time.sleep(wait)
                else:
                    answer = f"[回答失敗：{err[:100]}]"
                    break
        else:
            answer = "[已達重試上限，請稍後再試]"

    updated_history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return answer, confidence, sources, updated_history


import gradio as gr
import uuid as _uuid

# ── 問答功能 ──────────────────────────────────────────────────

def gradio_chat(user_message, history, session_state):
    print(f"📩 收到問題：{user_message}")  # 加這行
    if not user_message.strip():
        return history, history, session_state, ""
    if session_state is None:
        session_state = {"session_id": str(_uuid.uuid4())[:8], "turn": 1}

    rag_history = []
    for msg in history:
        rag_history.append({"role": msg["role"], "content": msg["content"]})

    answer, confidence, sources, _ = chat_rag(user_message, rag_history, k=3)

    source_text = "\n".join([f"・{s}" for s in sources]) if sources else "無"
    full_answer = (
        f"{answer}\n\n"
        f"---\n"
        f"📊 資料充足度：{confidence}　｜　參考文章 {len(sources)} 篇\n"
        f"📄 來源：\n{source_text}"
    )

    log_qa(session_state["session_id"], session_state["turn"],
           user_message, answer, sources, confidence)
    session_state["turn"] += 1
    history = history + [{"role": "user", "content": user_message}, {"role": "assistant", "content": full_answer}]
    return history, history, session_state, ""


def reset_chat(session_state):
    new_state = {"session_id": str(_uuid.uuid4())[:8], "turn": 1}
    return [], [], new_state, f"🔄 已清除對話記憶，新 Session ID：{new_state['session_id']}"


# ── 文章總覽功能 ──────────────────────────────────────────────

SENTIMENT_EMOJI = {"推薦": "👍", "普通": "😐", "負評": "👎", "失敗": "❓", "": "❓"}

def load_article_overview():
    """
    從 Google Sheet 讀取文章、摘要、情緒分析，合併後回傳 DataFrame。
    """
    try:
        posts_df   = read_sheet_df(ws_ptt,       PTT_HEADER)
        summary_df = read_sheet_df(ws_summary,   SUMMARY_HEADER)
        sent_df    = read_sheet_df(ws_sentiment, SENTIMENT_HEADER)

        # 合併摘要與情緒
        merged = posts_df[["post_id", "title", "url", "author", "date", "nrec"]].copy()
        merged = merged.merge(
            summary_df[["post_id", "summary"]], on="post_id", how="left"
        ).merge(
            sent_df[["post_id", "sentiment", "reason"]], on="post_id", how="left"
        )
        merged["summary"]   = merged["summary"].fillna("（尚未摘要）")
        merged["sentiment"] = merged["sentiment"].fillna("")
        merged["reason"]    = merged["reason"].fillna("")
        merged["評價"] = merged["sentiment"].apply(
            lambda s: f"{SENTIMENT_EMOJI.get(s, '❓')} {s}" if s else "❓ 未分析"
        )

        # 整理顯示欄位
        display_df = merged[["title", "author", "date", "nrec", "評價", "reason", "summary", "url"]].copy()
        display_df.columns = ["標題", "作者", "日期", "推文數", "評價", "評價理由", "摘要", "網址"]
        display_df = display_df.sort_values("推文數", ascending=False, key=lambda x: pd.to_numeric(x, errors="coerce"))
        return display_df
    except Exception as e:
        return pd.DataFrame([{"錯誤": str(e)}])


def filter_articles(sentiment_filter, search_keyword, overview_df_state):
    """
    依評價與關鍵字篩選文章。
    """
    df = overview_df_state.copy()
    if sentiment_filter and sentiment_filter != "全部":
        df = df[df["評價"].str.contains(sentiment_filter, na=False)]
    if search_keyword.strip():
        kw = search_keyword.strip()
        mask = (
            df["標題"].str.contains(kw, na=False) |
            df["摘要"].str.contains(kw, na=False)
        )
        df = df[mask]
    return df


# ── 建立 Gradio 介面（兩個分頁）────────────────────────────────

with gr.Blocks(title="PTT 電影版 RAG 問答") as demo:
    gr.Markdown("# 🎬 PTT 電影版 RAG 問答系統")

    with gr.Tabs():

        # ── 分頁一：多輪對話 ──────────────────────────────────
        with gr.TabItem("💬 問答對話"):
            gr.Markdown("根據 PTT 電影版文章回答問題，支援多輪對話與追問。")
            chatbot = gr.Chatbot(label="對話視窗", height=450, type="messages")
            history_state = gr.State([])
            session_state = gr.State(None)

            with gr.Row():
                user_input = gr.Textbox(
                    placeholder="輸入問題，例如：最近有哪些好看的電影？",
                    label="你的問題",
                    scale=5,
                    lines=1,
                )
                send_btn = gr.Button("送出", variant="primary", scale=1)

            with gr.Row():
                reset_btn = gr.Button("🔄 清除對話記憶", variant="secondary")
                status_text = gr.Textbox(label="狀態", interactive=False, scale=4)

            send_btn.click(
                fn=gradio_chat,
                inputs=[user_input, history_state, session_state],
                outputs=[chatbot, history_state, session_state, user_input],
            )
            user_input.submit(
                fn=gradio_chat,
                inputs=[user_input, history_state, session_state],
                outputs=[chatbot, history_state, session_state, user_input],
            )
            reset_btn.click(
                fn=reset_chat,
                inputs=[session_state],
                outputs=[chatbot, history_state, session_state, status_text],
            )

        # ── 分頁二：文章總覽 ──────────────────────────────────
        with gr.TabItem("📊 文章總覽"):
            gr.Markdown("從 Google Sheet 讀取所有文章，顯示摘要與推薦評價。")

            with gr.Row():
                sentiment_filter = gr.Dropdown(
                    choices=["全部", "👍 推薦", "😐 普通", "👎 負評", "❓ 未分析"],
                    value="全部",
                    label="篩選評價",
                    scale=1,
                )
                search_input = gr.Textbox(
                    placeholder="輸入關鍵字搜尋標題或摘要...",
                    label="關鍵字搜尋",
                    scale=3,
                )
                refresh_btn = gr.Button("🔄 重新載入", variant="secondary", scale=1)

            overview_table = gr.Dataframe(
                label="文章列表（依推文數排序）",
                wrap=True,
                interactive=False,
            )
            overview_df_state = gr.State(pd.DataFrame())

            def on_refresh():
                df = load_article_overview()
                return df, df

            def on_filter(sentiment_filter, search_keyword, overview_df_state):
                filtered = filter_articles(sentiment_filter, search_keyword, overview_df_state)
                return filtered

            refresh_btn.click(
                fn=on_refresh,
                outputs=[overview_table, overview_df_state],
            )
            sentiment_filter.change(
                fn=on_filter,
                inputs=[sentiment_filter, search_input, overview_df_state],
                outputs=[overview_table],
            )
            search_input.submit(
                fn=on_filter,
                inputs=[sentiment_filter, search_input, overview_df_state],
                outputs=[overview_table],
            )

            # 開啟介面時自動載入資料
            demo.load(
                fn=on_refresh,
                outputs=[overview_table, overview_df_state],
            )

print("✅ Gradio 介面已定義，執行下一格啟動。")


✅ Gradio 介面已定義，執行下一格啟動。


/tmp/ipykernel_2138/1094627479.py:147: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="對話視窗", height=450, type="messages")


In [ ]:
import google.generativeai as genai

genai.configure(api_key=api_key)
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print("✅ 已重設 Gemini 連線")

# 立刻測試
try:
    r = llm.generate_content("測試")
    print("✅ 回應：", r.text)
except Exception as e:
    print("❌ 錯誤：", str(e))

✅ 已重設 Gemini 連線


## 12. 啟動問答


In [26]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")
print("✅ now_iso 已定義")

✅ now_iso 已定義


In [27]:
# 🆕 Gradio 介面（推薦）
# share=True 會產生公開連結，方便在 Colab 外部開啟
demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4944bb24e32055613b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📩 收到問題：推薦我一些喜劇片
🔄 呼叫 Gemini（第 1 次）...
✅ 取得回答
📩 收到問題：最近有哪些恐怖片上映
🔄 呼叫 Gemini（第 1 次）...
✅ 取得回答
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4944bb24e32055613b.gradio.live


In [ ]:
# 原本的 terminal 多輪對話模式（備用）
# start_chat(k=3)

# 單次問答模式（備用）
# question = input("請輸入問題：")
# answer, confidence, sources = query_rag(question, k=3)
# print(answer)


## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet。
7. 🆕 **Gradio**：執行啟動格後，點擊 `Running on public URL` 的連結即可開啟介面。
8. 🆕 **429 限流**：摘要與情緒分析請用 `delay=13`；問答功能已內建自動重試。
